In [10]:
import ollama
print('ollama import ok')

ollama import ok


In [11]:
import os
import ollama

dataset = []
file_path = 'cricket_150_lines.txt'

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        dataset = file.readlines()
        print(f'Loaded {len(dataset)} entries')
else:
    print(f'Error: File "{file_path}" not found in {os.getcwd()}')
    print('Please check the file path and ensure the file exists.')

Loaded 150 entries


In [12]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

In [13]:
VECTOR_DB = []

In [14]:
def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))


for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')

Added chunk 1/150 to the database
Added chunk 2/150 to the database
Added chunk 3/150 to the database
Added chunk 4/150 to the database
Added chunk 5/150 to the database
Added chunk 6/150 to the database
Added chunk 7/150 to the database
Added chunk 8/150 to the database
Added chunk 9/150 to the database
Added chunk 10/150 to the database
Added chunk 11/150 to the database
Added chunk 12/150 to the database
Added chunk 13/150 to the database
Added chunk 14/150 to the database
Added chunk 15/150 to the database
Added chunk 16/150 to the database
Added chunk 17/150 to the database
Added chunk 18/150 to the database
Added chunk 19/150 to the database
Added chunk 20/150 to the database
Added chunk 21/150 to the database
Added chunk 22/150 to the database
Added chunk 23/150 to the database
Added chunk 24/150 to the database
Added chunk 25/150 to the database
Added chunk 26/150 to the database
Added chunk 27/150 to the database
Added chunk 28/150 to the database
Added chunk 29/150 to the dat

In [15]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [16]:
def get_embedding(text):
  response = ollama.embed(model=EMBEDDING_MODEL, input=text)
  embeddings = response.get('embeddings', [])
  if not embeddings:
    raise ValueError('Ollama returned no embedding. Please enter a non-empty question.')
  return embeddings[0]


def retrieve(query, top_n=3):
  query = query.strip()
  if not query:
    raise ValueError('Please enter a question before retrieving knowledge.')
  query_embedding = get_embedding(query)
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]

input_query = input('Ask me a question: ').strip()

try:
  retrieved_knowledge = retrieve(input_query)

  print('Retrieved knowledge:')
  for chunk, similarity in retrieved_knowledge:
    print(f' - (similarity: {similarity:.2f}) {chunk}')

  ## Now we can use the retrieved knowledge to construct a prompt for the language model. The prompt will include the retrieved chunks as context, and then we will ask the model to answer the user's question based on that context.
  instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''
except ValueError as error:
  retrieved_knowledge = []
  instruction_prompt = ''
  print(f'Error: {error}')

Retrieved knowledge:
 - (similarity: 0.76) 101. Cricket is one of the most popular sports in the world, enjoyed for its excitement, teamwork, and competitive spirit.

 - (similarity: 0.76) 146. Cricket is one of the most popular sports in the world, enjoyed for its excitement, teamwork, and competitive spirit.

 - (similarity: 0.76) 102. Cricket is one of the most popular sports in the world, enjoyed for its excitement, teamwork, and competitive spirit.



In [17]:
if not instruction_prompt or not input_query:
  print('Ask a non-empty question and run the retrieval cell first.')
else:
  stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
      {'role': 'system', 'content': instruction_prompt},
      {'role': 'user', 'content': input_query},
    ],
    stream=True,
  )

  # print the response from the chatbot in real-time
  print('Chatbot response:')
  for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)

Chatbot response:
Cricket is a team sport played with a hard ball and bat on a rectangular field with two sets of three stumps (called wickets) at each end. The objective is to score runs by hitting the ball with the bat and then running between the wickets, while the opposing team tries to stop them by getting batsmen out.

In general, cricket involves several key aspects:

* 11 players from both teams play on each side of the field.
* Each player has two chances (called "overs") to hit the ball or make a score.
* The batting team sends one player at a time to bat until all nine players are out.
* The bowling team tries to get batsmen out by delivering the ball in different ways and angles.

Cricket is often played on a field with markings, such as three stumps (wickets) and two bails or crests. The game can take several forms, including Test cricket, One-Day Internationals (ODIs), Twenty20 International (T20I), and domestic competitions like county matches in England.

The players wh